# FLOWIN UMaine LCOE Analysis: 1 GW wind farm with 20-MW WTGs in the New York Bight

If you want to run this notebook, keep in mind that the library path from the FLORIS config file has to be changed accordingly to your new path

## Imports and environment set up

In [1]:
from pathlib import Path
from itertools import product

import pandas as pd
import matplotlib.pyplot as plt

from whale import Project
from whale.utilities import load_yaml

pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.max_columns = 100
pd.options.display.max_rows = 100

%load_ext autoreload
%autoreload 2

# Function to Extract Results

In [2]:
def extract_results(project_name):  # Run the project and clean up the logging
    #project_name = "UMaine_catenary"
    library_path = Path("library").resolve()
    
    metrics_configuration = {
        "# Turbines": {"metric": "n_turbines", "kwargs": {}},
        "Turbine Rating (MW)": {"metric": "turbine_rating", "kwargs": {}},
        "Project Capacity (MW)": {"metric": "capacity", "kwargs": {"units": "mw"}},
        "# OSS": {"metric": "n_substations", "kwargs": {}},
        "Total Export Cable Length (km)": {"metric": "export_system_total_cable_length", "kwargs": {}},
        "Total Array Cable Length (km)": {"metric": "array_system_total_cable_length", "kwargs": {}},
        "CapEx ($)": {"metric": "capex", "kwargs": {}},
        "CapEx per kW ($/kW)": {"metric": "capex", "kwargs": {"per_capacity": "kw"}},
        "OpEx ($)": {"metric": "opex", "kwargs": {}},
        "OpEx per kW ($/kW)": {"metric": "opex", "kwargs": {"per_capacity": "kw"}},
        "AEP (MWh)": {
            "metric": "energy_production",
            "kwargs": {"units": "mw", "aep": True, "with_losses": True}
        },
        "AEP per kW (MWh/kW)": {
            "metric": "energy_production",
            "kwargs": {"units": "mw", "per_capacity": "kw", "aep": True, "with_losses": True}
        },
        "Net Capacity Factor Without Unmodeled Losses (%)": {
            "metric": "capacity_factor",
            "kwargs": {"which": "net"}
        },
        "Net Capacity Factor With All Losses (%)": {
            "metric": "capacity_factor",
            "kwargs": {"which": "net", "with_losses": True}
        },
        "Gross Capacity Factor (%)": {"metric": "capacity_factor", "kwargs": {"which": "gross"}},
        "Energy Availability (%)": {"metric": "availability", "kwargs": {"which": "energy"}},
        "LCOE ($/MWh)": {"metric": "lcoe", "kwargs": {}},
        "IRR (%)": {"metric": "irr", "kwargs": {}},
        "NPV ($)": {"metric": "npv", "kwargs": {}},
    }

    metrics_order = [
        "# Turbines",
        "Turbine Rating (MW)",
        "Project Capacity (MW)",
        "# OSS",
        "Total Export Cable Length (km)",
        "Total Array Cable Length (km)",
        "FCR (%)",
        "Offtake Price ($/MWh)",
        "CapEx ($)",
        "CapEx per kW ($/kW)",
        "System CapEx for Export Cables ($)",
        "Installation CapEx for Export Cables ($)",
        "CapEx Without Export Cables ($)",
        "OpEx ($)",
        "OpEx per kW ($/kW)",
        "Annual OpEx per kW ($/kW)",
        "Energy Availability (%)",
        "Gross Capacity Factor (%)",
        "Net Capacity Factor Without Unmodeled Losses (%)",
        "Net Capacity Factor With All Losses (%)",
        "AEP (MWh)",
        "AEP per kW (MWh/kW)",
        "LCOE ($/MWh)",
        "IRR (%)",
        "NPV ($)",
    ]
    final_cols = ["CapEx ($)", "OpEx ($)", "Energy Production (GWh)", "Revenue ($)", "Cash Flow ($)"]
    
    config = load_yaml(
        library_path / "project/config",
        f"{project_name.replace(' ', '_')}_base.yaml"
    )
    project = Project(
        # Basic Model Configurations
        library_path=library_path,
        #weather_profile=library_path / "weather" / "ocean_wind_1_39.0_-74.0_1959_2023.csv",
        connect_floris_to_layout=True,
        connect_orbit_array_design=True,
        **config,
    )

    
    project.run(
        which_floris="wind_rose",
        full_wind_rose=False,
        floris_reinitialize_kwargs=dict(cut_in_wind_speed=3.0, cut_out_wind_speed=25.0)
    )
    project.wombat.env.cleanup_log_files()  # Delete logging data from WOMBAT
    
    fig, ax = project.plot_farm(return_fig=True)
    
    #ORBIT breakdown

    print(project.orbit.capex_breakdown)
    
    # Gather the high-level results
    report_df = project.generate_report(metrics_configuration, project_name).T
    export_system = project.orbit.system_costs["ExportCableInstallation"]
    export_installation = project.orbit.installation_costs["ExportCableInstallation"]
    capex_sans_export = project.orbit.total_capex - export_system - export_installation
    additional_reporting = pd.DataFrame(
        [
            ["FCR (%)", project.fixed_charge_rate],
            ["Offtake Price ($/MWh)", project.offtake_price],
            ["System CapEx for Export Cables ($)", export_system],
            ["Installation CapEx for Export Cables ($)", export_installation],
            ["CapEx Without Export Cables ($)", capex_sans_export],
            ["Annual OpEx per kW ($/kW)", report_df.loc["OpEx per kW ($/kW)", project_name] / project.operations_years],
        ],
        columns=["Project"] + report_df.columns.tolist(),
    ).set_index("Project")
    report_df = pd.concat((report_df, additional_reporting), axis=0).loc[metrics_order]

    # Gather the detailed results
    monthly_results = project.cash_flow(breakdown=True).join(project.energy_production(frequency="month-year")).fillna(0)
    monthly_results = monthly_results.assign(
        CapEx_Installation=monthly_results[[c for c in monthly_results if c.startswith("CapEx") and c.endswith("Installation")]].sum(axis=1),
        CapEx_System=monthly_results[[c for c in monthly_results if c.startswith("CapEx") and c.endswith("System")]].sum(axis=1),
    )

    # monthly_results.to_csv(library_path / "results" / f"{project_name.lower().replace(' ', '_')}_monthly_detailed_results.csv")
    monthly_results["CapEx ($)"] = monthly_results[["CapEx_Installation", "CapEx_Soft", "CapEx_Project", "CapEx_System", "CapEx_Turbine"]].sum(axis=1)
    monthly_results = monthly_results.rename(columns={"OpEx": "OpEx ($)","Revenue": "Revenue ($)", "cash_flow": "Cash Flow ($)"})[final_cols]

    if "ExportSystemDesign" in project.orbit._phases:
        export = "ExportSystemDesign"
    else:
        export = "ElectricalDesign"

    # Create the inputs data
    inputs = pd.DataFrame(
        [
            ["FCR", project.fixed_charge_rate],
            ["Discount rate (%)", project.discount_rate],
            ["# Turbines", project.n_turbines()],
            ["Turbine Rating (MW)", project.turbine_rating()],
            ["Project Capacity (MW)", project.capacity("mw")],
            ["# OSS", project.n_substations()],
            ["Substructure type", "??"],
            ["Row spacing (rotor diameters)", "not used for custom layouts"],
            ["Turbine spacing (rotor diameters)", "not used for custom layouts"],
            ["Depth (m)", project.orbit.config["site"]["depth"]],
            [
                "Mean wind speed (m/s)",
                project.weather.loc[
                    project.orbit_start_date: project.wombat.env.end_datetime,
                    "windspeed_100m"
                ].mean()
            ],
            ["Distance to landfall (km)", project.orbit.config["site"]["distance_to_landfall"]],
            ["Distance to port (km)", project.wombat.config.port_distance],
            ["Interconnection distance (km)", project.orbit._phases[export]._distance_to_interconnection],
            ["# of POIs", "??"],
            ["Export cable type", [*project.orbit._phases[export].cable_lengths_by_type]],
            ["Array cable type", [*project.orbit._phases["CustomArraySystemDesign"].cable_lengths_by_type]],
        ],
        columns=["Inputs", f"{project_name}"]
    ).set_index("Inputs")

    # Save the outputs
    report_df.index.name = "Metrics"
    wrong_indexes = ["Offtake Price ($/MWh)", "System CapEx for Export Cables ($)", "Installation CapEx for Export Cables ($)", "CapEx Without Export Cables ($)", "IRR (%)", "NPV ($)"]
    
    
    return report_df.drop(wrong_indexes,axis=0)

# Extract CapEx, OpEx, and AEP Outputs - Multiple Runs
Availability and OpEx varies depending on the run as it uses probabilistic functions, so we collect the data from x runs and calculate the average OpEx and AEP

In [3]:
def display_results_all_runs(n_runs):
    import pandas as pd
    
    df_list_fixed_bottom = list()
    df_list_floating = list()
    df_final_list = list()
    
    for i in range(0,n_runs):
        df_2 = extract_results("New_York_Bight")
        #display(df_2)
        #final_df = pd.concat(df_final_list, axis=1)
        
        if i == 0:
            df_list_floating = df_2
            #display(df_list_semitaut)
        else:
            df_list_floating = pd.concat([df_list_floating, df_2], axis=1)
            #display(df_list_semitaut)      
    
    #for i in range(0,n_runs):
        #df_1 = extract_results("COE_2022_fixed_bottom")
        #display(df_1)
        #final_df = pd.concat(df_final_list, axis=1)
        
        #if i == 0:
            #df_list_fixed_bottom = df_1
            #display(df_list_catenary)
        #else:
            #df_list_fixed_bottom = pd.concat([df_list_fixed_bottom, df_1], axis=1)
            #display(df_list_catenary)
            
            

            
    
    #final_df = pd.concat([df_list_fixed_bottom, df_list_floating], axis=1)
    final_df = df_list_floating
                          
    return final_df

import time
t1 = time.perf_counter()

summary_results = display_results_all_runs(1)

t2 = time.perf_counter()
print('time taken to run:',t2-t1)




ORBIT library intialized at 'C:\test_before_PTO\WHaLE\examples\library'


Missing data in columns ['bury_speed']; all values will be calculated.UserWarning: C:\Users\dmulash\.conda\envs\test_before_PTO\lib\site-packages\ORBIT\phases\design\array_system_design.py:906
Missing data in columns ['bury_speed']; all values will be calculated.

Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
Correcting negative Overhang:-2.5
time taken to run: 758.8602377000207


In [4]:
display(summary_results)

,New_York_Bight
Metrics,
# Turbines,50.0000
Turbine Rating (MW),20.0000
Project Capacity (MW),"1,000.0000"
# OSS,1.0000
Total Export Cable Length (km),178.3401
Total Array Cable Length (km),333.0854
FCR (%),0.0569
CapEx ($),"3,982,313,735.7458"
CapEx per kW ($/kW),"3,982.3137"


In [5]:
summary_results.to_csv('library/results/LCOE_summary_results.csv')

In [6]:
def get_average_data(summary_results):
    summary_results.groupby(by=summary_results.columns, axis=1).mean()
    return summary_results

display(get_average_data(summary_results))

,New_York_Bight
Metrics,
# Turbines,50.0000
Turbine Rating (MW),20.0000
Project Capacity (MW),"1,000.0000"
# OSS,1.0000
Total Export Cable Length (km),178.3401
Total Array Cable Length (km),333.0854
FCR (%),0.0569
CapEx ($),"3,982,313,735.7458"
CapEx per kW ($/kW),"3,982.3137"
